In [17]:
# Install NLTK for text processing
!pip -q install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [14]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import gradio as gr
import re
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
data = {
    "disease": [
        "Common Cold",
        "Influenza",
        "Migraine",
        "Food Poisoning",
        "Gastroenteritis",
        "Allergy",
        "Asthma",
        "Tension Headache",
        "Sinusitis",
        "Urinary Tract Infection",
        "COVID-like Respiratory Infection",
        "Dehydration"
    ],

    "symptoms": [
        "runny nose sneezing sore throat cough mild fever congestion",
        "fever chills cough sore throat body ache fatigue headache",
        "severe headache nausea vomiting sensitivity to light sensitivity to sound",
        "nausea vomiting diarrhea stomach pain abdominal cramps fever",
        "diarrhea vomiting nausea stomach pain abdominal cramps weakness",
        "sneezing runny nose itchy eyes watery eyes congestion",
        "shortness of breath wheezing chest tightness cough",
        "headache pressure stress neck pain muscle tension",
        "facial pain nasal congestion headache runny nose fever",
        "burning urination frequent urination lower abdominal pain cloudy urine",
        "fever cough sore throat fatigue body ache loss of smell breathing difficulty",
        "thirst dry mouth weakness dizziness fatigue dark urine"
    ],

    "severity": [
        "Low",
        "Moderate",
        "Moderate",
        "Moderate",
        "Moderate",
        "Low",
        "High",
        "Low",
        "Moderate",
        "Moderate",
        "Moderate",
        "Moderate"
    ],

    "concerns": [
        "Usually mild, but symptoms that become severe or persistent should be evaluated.",
        "High fever, severe weakness, breathing difficulty or worsening symptoms need medical evaluation.",
        "A sudden extremely severe headache, neurological symptoms or fainting requires urgent evaluation.",
        "Severe dehydration, blood in stool, persistent vomiting or severe abdominal pain require medical attention.",
        "Persistent vomiting, severe dehydration, blood in stool or severe abdominal pain require medical attention.",
        "Severe breathing difficulty or swelling of the face/throat can be an emergency.",
        "Severe breathing difficulty, inability to speak normally or blue/gray lips require emergency care.",
        "A sudden severe headache, neurological symptoms or headache after injury requires medical evaluation.",
        "Severe facial swelling, severe headache, eye swelling or worsening symptoms require medical evaluation.",
        "Fever, back/flank pain, vomiting or worsening urinary symptoms require medical evaluation.",
        "Breathing difficulty, persistent chest pain, confusion or severe weakness require urgent medical evaluation.",
        "Severe dizziness, confusion, fainting or inability to keep fluids down requires urgent medical attention."
    ],

    "self_care": [
        "Rest, drink fluids and monitor symptoms.",
        "Rest, maintain hydration and monitor fever and breathing.",
        "Rest in a quiet dark environment and stay hydrated.",
        "Drink fluids or oral rehydration solution and eat light foods as tolerated.",
        "Maintain hydration and eat light foods as tolerated.",
        "Avoid known triggers and keep the environment free from irritants.",
        "Avoid known triggers and follow an existing clinician-provided asthma plan.",
        "Rest, hydrate and consider reducing stress and prolonged screen/neck strain.",
        "Rest, hydrate and monitor symptoms.",
        "Drink adequate fluids and seek medical advice if symptoms worsen.",
        "Rest, hydrate and monitor respiratory symptoms.",
        "Increase fluids gradually and monitor urine color and dizziness."
    ]
}

df = pd.DataFrame(data)

df

,disease,symptoms,severity,concerns,self_care
0,Common Cold,runny nose sneezing sore throat cough mild fev...,Low,"Usually mild, but symptoms that become severe ...","Rest, drink fluids and monitor symptoms."
1,Influenza,fever chills cough sore throat body ache fatig...,Moderate,"High fever, severe weakness, breathing difficu...","Rest, maintain hydration and monitor fever and..."
2,Migraine,severe headache nausea vomiting sensitivity to...,Moderate,"A sudden extremely severe headache, neurologic...",Rest in a quiet dark environment and stay hydr...
3,Food Poisoning,nausea vomiting diarrhea stomach pain abdomina...,Moderate,"Severe dehydration, blood in stool, persistent...",Drink fluids or oral rehydration solution and ...
4,Gastroenteritis,diarrhea vomiting nausea stomach pain abdomina...,Moderate,"Persistent vomiting, severe dehydration, blood...",Maintain hydration and eat light foods as tole...
5,Allergy,sneezing runny nose itchy eyes watery eyes con...,Low,Severe breathing difficulty or swelling of the...,Avoid known triggers and keep the environment ...
6,Asthma,shortness of breath wheezing chest tightness c...,High,"Severe breathing difficulty, inability to spea...",Avoid known triggers and follow an existing cl...
7,Tension Headache,headache pressure stress neck pain muscle tension,Low,"A sudden severe headache, neurological symptom...","Rest, hydrate and consider reducing stress and..."
8,Sinusitis,facial pain nasal congestion headache runny no...,Moderate,"Severe facial swelling, severe headache, eye s...","Rest, hydrate and monitor symptoms."
9,Urinary Tract Infection,burning urination frequent urination lower abd...,Moderate,"Fever, back/flank pain, vomiting or worsening ...",Drink adequate fluids and seek medical advice ...


In [16]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Combine all symptoms into a single string and tokenize
all_symptoms = ' '.join(df['symptoms'].values)
symptom_tokens = nltk.word_tokenize(all_symptoms)

# Count the frequency of each symptom
symptom_counts = Counter(symptom_tokens)

# Remove stop words and common non-symptom words if necessary (example)
# You might want to refine this list based on your data
stop_words = set(nltk.corpus.stopwords.words('english')) # Not used here, as TfidfVectorizer handles this for the model, but good practice for raw text analysis
common_nonsymptoms = {'and', 'or', 'a', 'to', 'in', 'with', 'the', 'of'}

filtered_symptom_counts = {word: count for word, count in symptom_counts.items() if word not in common_nonsymptoms and len(word) > 2}

# Get the top N most common symptoms
top_n = 10 # Adjusted to 10 as per user request
top_symptoms = Counter(filtered_symptom_counts).most_common(top_n)

# Prepare data for plotting
symptoms_plot_df = pd.DataFrame(top_symptoms, columns=['Symptom', 'Frequency'])

# Create the bar plot
plt.figure(figsize=(12, 7))
sns.barplot(x='Frequency', y='Symptom', data=symptoms_plot_df, palette='viridis')
plt.title('Top 10 Symptom Frequency Distribution')
plt.xlabel('Frequency')
plt.ylabel('Symptom')
plt.tight_layout()
plt.show()

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [5]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

symptom_matrix = vectorizer.fit_transform(df["symptoms"])

print("Model prepared successfully!")
print("Number of conditions:", len(df))

Model prepared successfully!
Number of conditions: 12


In [6]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
emergency_symptoms = {
    "difficulty breathing",
    "shortness of breath",
    "severe chest pain",
    "chest pain",
    "fainting",
    "unconscious",
    "confusion",
    "blue lips",
    "blue skin",
    "severe bleeding",
    "coughing blood",
    "vomiting blood",
    "blood in stool",
    "seizure",
    "sudden weakness",
    "face drooping",
    "slurred speech",
    "severe allergic reaction"
}

In [8]:
def check_emergency(text):

    text = clean_text(text)

    detected = []

    for symptom in emergency_symptoms:

        if symptom in text:
            detected.append(symptom)

    return detected

In [9]:
def predict_conditions(user_symptoms, top_n=3):

    user_symptoms = clean_text(user_symptoms)

    if not user_symptoms:
        return []

    user_vector = vectorizer.transform([user_symptoms])

    similarity_scores = cosine_similarity(
        user_vector,
        symptom_matrix
    )[0]

    top_indices = np.argsort(similarity_scores)[::-1][:top_n]

    results = []

    for index in top_indices:

        score = similarity_scores[index]

        results.append({
            "disease": df.iloc[index]["disease"],
            "score": round(float(score) * 100, 2),
            "severity": df.iloc[index]["severity"],
            "concerns": df.iloc[index]["concerns"],
            "self_care": df.iloc[index]["self_care"]
        })

    return results

In [10]:
test_symptoms = "fever cough sore throat body ache"

results = predict_conditions(test_symptoms)

for result in results:
    print(
        result["disease"],
        "| Match:",
        result["score"],
        "%",
        "| Severity:",
        result["severity"]
    )

Influenza | Match: 79.37 % | Severity: Moderate
COVID-like Respiratory Infection | Match: 65.68 % | Severity: Moderate
Common Cold | Match: 45.85 % | Severity: Low


In [11]:
def health_assistant(symptoms):

    if not symptoms or len(symptoms.strip()) < 3:

        return """
### Please enter your symptoms

Example:

`fever, cough, sore throat and body ache`
"""

    # Check emergency symptoms
    emergencies = check_emergency(symptoms)

    # Get possible conditions
    results = predict_conditions(symptoms, top_n=3)

    output = ""

    # Emergency warning
    if emergencies:

        output += """
# 🚨 URGENT WARNING

The symptoms you entered include potentially serious warning signs:

"""

        for item in emergencies:
            output += f"- **{item.title()}**\n"

        output += """

Please seek urgent medical attention or contact your local emergency service.

Do not rely on this application for emergency decisions.

---

"""

    output += "# 🩺 Possible Health Conditions\n\n"

    output += (
        "**Important:** These are possible matches based on the "
        "symptoms entered. This is NOT a medical diagnosis.\n\n"
    )

    for i, result in enumerate(results, start=1):

        output += f"""
## {i}. {result['disease']}

**Symptom match:** {result['score']}%

**General severity:** {result['severity']}

### ⚠️ Concerns
{result['concerns']}

### 💡 General self-care
{result['self_care']}

---

"""

    output += """
# 👨‍⚕️ When should you contact a doctor?

Consider speaking with a qualified healthcare professional if:

- Symptoms are severe
- Symptoms are getting worse
- Symptoms persist or repeatedly return
- You have significant difficulty breathing
- You develop severe pain
- You cannot keep fluids down
- You feel faint, confused or unusually weak
- You are concerned about your symptoms

### Disclaimer

This application is an educational prototype and is not a substitute
for professional medical diagnosis, treatment or emergency care.
"""

    return output

In [12]:
print(
    health_assistant(
        "I have fever, cough, sore throat and body ache"
    )
)

# 🩺 Possible Health Conditions

**Important:** These are possible matches based on the symptoms entered. This is NOT a medical diagnosis.


## 1. Influenza

**Symptom match:** 79.37%

**General severity:** Moderate

### ⚠️ Concerns
High fever, severe weakness, breathing difficulty or worsening symptoms need medical evaluation.

### 💡 General self-care
Rest, maintain hydration and monitor fever and breathing.

---


## 2. COVID-like Respiratory Infection

**Symptom match:** 65.68%

**General severity:** Moderate

### ⚠️ Concerns
Breathing difficulty, persistent chest pain, confusion or severe weakness require urgent medical evaluation.

### 💡 General self-care
Rest, hydrate and monitor respiratory symptoms.

---


## 3. Common Cold

**Symptom match:** 45.85%

**General severity:** Low

### ⚠️ Concerns
Usually mild, but symptoms that become severe or persistent should be evaluated.

### 💡 General self-care
Rest, drink fluids and monitor symptoms.

---


# 👨‍⚕️ When should you contact a d

In [13]:
with gr.Blocks(
    title="Health Symptom Assistant"
) as app:

    gr.Markdown(
        """
        # 🩺 Health Symptom Assistant

        Enter your symptoms and the application will show
        possible health conditions, general concerns and
        recommended next steps.

        ⚠️ **This application does not provide a medical diagnosis.**
        """
    )

    with gr.Row():

        with gr.Column():

            symptoms_input = gr.Textbox(
                label="Enter your symptoms",
                placeholder=(
                    "Example: fever, cough, headache, "
                    "sore throat..."
                ),
                lines=6
            )

            analyze_button = gr.Button(
                "🔍 Analyze Symptoms",
                variant="primary"
            )

        with gr.Column():

            output = gr.Markdown(
                """
                ### Results will appear here

                Enter your symptoms and click
                **Analyze Symptoms**.
                """
            )

    analyze_button.click(
        fn=health_assistant,
        inputs=symptoms_input,
        outputs=output
    )

    gr.Markdown(
        """
        ---

        **Medical Disclaimer:**
        This is an educational software prototype.
        It should not be used to diagnose or treat medical conditions.
        For serious or worsening symptoms, consult a qualified
        healthcare professional or emergency service.
        """
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62759606330323a922.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
